# 14 String Functions

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Clean, reshape and extract text with Spark's built-in string functions: trimming and case, concatenation, substrings, padding, splitting, and regular expressions (<code>regexp_replace</code>, <code>regexp_extract</code>, <code>rlike</code>).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Raw data is messy text: names with extra spaces, phone numbers in five formats, product codes hidden inside descriptions, CSV fields packed into one column. Cleaning text is a large part of every Silver-layer job. Spark has over 50 built-in string functions, and using them (instead of Python UDFs) keeps pipelines fast.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The main families</b>
<ul><li><b>Cleaning:</b> <code>trim</code>, <code>ltrim</code>, <code>rtrim</code>, <code>lower</code>, <code>upper</code>, <code>initcap</code></li><li><b>Combining:</b> <code>concat</code> (null if any input is null), <code>concat_ws</code> (skips nulls, adds a separator)</li><li><b>Slicing:</b> <code>substring(col, pos, len)</code> (1-based), <code>left</code>, <code>right</code>, <code>length</code></li><li><b>Padding and formatting:</b> <code>lpad</code>, <code>rpad</code>, <code>format_string</code>, <code>format_number</code></li><li><b>Splitting:</b> <code>split</code> (returns an array), then <code>[i]</code>, <code>element_at</code>, or <code>explode</code></li><li><b>Searching:</b> <code>contains</code>, <code>startswith</code>, <code>instr</code>, <code>locate</code>, <code>like</code>, <code>rlike</code></li><li><b>Regex:</b> <code>regexp_replace</code> (substitute), <code>regexp_extract</code> (capture a group), <code>regexp_extract_all</code></li><li><b>Others:</b> <code>translate</code> (character-by-character), <code>reverse</code>, <code>repeat</code>, <code>soundex</code>, <code>levenshtein</code></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A telecom imports customer records from three regional CRMs. Phone numbers arrive as <code>"+91 98765-43210"</code>, <code>"(044) 2345 6789"</code> and <code>"9876543210 "</code>. Names arrive as <code>"  RAJ, amal "</code>. The Silver job trims and title-cases names, splits "last, first", strips every non-digit from phones with <code>regexp_replace</code>, keeps the last 10 digits, and extracts the area code with <code>regexp_extract</code>. That is all native string functions, with no UDF.
</div>

## Syntax

```python
from pyspark.sql import functions as F

F.trim("name"), F.upper("name"), F.initcap("name")
F.concat_ws(" ", "first", "last")
F.substring("code", 1, 3)                    # 1-based position
F.split("full_name", ",")[0]
F.regexp_replace("phone", r"[^0-9]", "")     # remove non-digits
F.regexp_extract("email", r"@(.+)$", 1)      # capture group 1
F.lpad("id", 6, "0")                         # 42 -> 000042
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates messy customer records: names with odd spacing and case, phone numbers in different formats, emails and product codes.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 5-row table. Row 4 has a null email, and row 5 has a null middle name.
</div>

In [0]:
from pyspark.sql import functions as F

raw = spark.createDataFrame(
    [
        (1, "  raj, AMAL ", "Kumar", "+91 98765-43210", "Amal.Raj@Example.com", "SKU-IN-00042-BLUE"),
        (2, "smith,john", "Paul", "(044) 2345 6789", "john_smith@corp.co.uk", "SKU-UK-01337-RED"),
        (3, "NAIR , vivek", "", "9876543210 ", "vivek@mail.ae", "SKU-AE-00007-GREEN"),
        (4, "iyer,Priya", "Lakshmi", "+1-415-555-0132", None, "SKU-US-12345-BLACK"),
        (5, "ali, sara", None, "  0501234567", "SARA@MAIL.AE ", "bad code"),
    ],
    "id INT, full_name STRING, middle_name STRING, phone STRING, email STRING, product_code STRING",
)
display(raw)

## 1. Trimming and changing case

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Removes surrounding spaces and normalizes case with <code>trim</code>, <code>lower</code>, <code>upper</code> and <code>initcap</code>.<br><br>
<b>Why it matters</b><br>Matching and deduplication fail on <code>"Amal "</code> versus <code>"amal"</code>. Normalizing is usually the very first cleaning step.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Trimmed names in lower, upper and title case. <code>length</code> shows that trimming removed the spaces from row 1 (12 characters down to 9). Notice that <code>initcap</code> only capitalizes letters that follow a <b>space</b>: <code>smith,john</code> becomes <code>Smith,john</code>. That is why section 2 splits the name first.
</div>

In [0]:
raw.select(
    "id",
    F.trim("full_name").alias("trimmed"),
    F.lower(F.trim("full_name")).alias("lower"),
    F.upper(F.trim("full_name")).alias("upper"),
    F.initcap(F.trim("full_name")).alias("initcap"),
    F.length("full_name").alias("len_raw"),
    F.length(F.trim("full_name")).alias("len_trimmed"),
).show(truncate=False)

## 2. Splitting "last, first" into columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Splits the name on the comma, trims each part, and title-cases it.<br><br>
<b>Why it matters</b><br><code>split</code> returns an <b>array</b>. Index it with <code>[0]</code>, <code>[1]</code> (0-based) or <code>element_at(arr, 1)</code> (1-based). The separator is a <b>regular expression</b>, so <code>\s*,\s*</code> handles spaces around the comma.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Clean <code>first_name</code> and <code>last_name</code> columns, such as <code>Amal</code> and <code>Raj</code> for row 1, and <code>Vivek</code> and <code>Nair</code> for row 3.
</div>

In [0]:
parts = F.split(F.trim("full_name"), r"\s*,\s*")

names = raw.select(
    "id",
    F.initcap(parts[1]).alias("first_name"),
    F.initcap(parts[0]).alias("last_name"),
    F.size(parts).alias("n_parts"),
)
names.show()

## 3. Concatenation and null behavior

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a display name with <code>concat</code> and with <code>concat_ws</code>, using a middle name that is sometimes empty or null.<br><br>
<b>Why it matters</b><br><code>concat</code> returns <b>null if any input is null</b>, which silently blanks out rows. <code>concat_ws</code> skips nulls (but not empty strings). Choosing the wrong one is a common data bug.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For row 5 (null middle name), <code>concat</code> gives null while <code>concat_ws</code> gives <code>Sara Ali</code>. For row 3 (empty middle name), <code>concat_ws</code> gives a double space unless you turn empty strings into null first.
</div>

In [0]:
with_names = raw.join(names, "id")

with_names.select(
    "id",
    F.concat("first_name", F.lit(" "), "middle_name", F.lit(" "), "last_name").alias("concat"),
    F.concat_ws(" ", "first_name", "middle_name", "last_name").alias("concat_ws"),
    # nullif turns empty strings into null, so concat_ws skips them too
    F.concat_ws(" ", "first_name", F.expr("nullif(trim(middle_name), '')"), "last_name").alias("clean_full_name"),
).orderBy("id").show(truncate=False)

## 4. Substrings, padding and formatting

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Extracts parts of fixed-format codes with <code>substring</code>, pads IDs with <code>lpad</code>, and formats text with <code>format_string</code>.<br><br>
<b>Why it matters</b><br>Legacy systems use fixed-width codes ("characters 5 to 6 are the country"). Downstream systems often need zero-padded IDs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>country</code> is <code>IN</code>, <code>UK</code>, <code>AE</code>, <code>US</code> for well-formed codes (row 5 gives junk, see section 6). <code>padded_id</code> is like <code>CUST-000001</code>.
</div>

In [0]:
raw.select(
    "product_code",
    F.substring("product_code", 5, 2).alias("country"),      # positions are 1-based
    F.expr("left(product_code, 3)").alias("prefix"),
    F.expr("right(product_code, 4)").alias("last4"),
    F.concat(F.lit("CUST-"), F.lpad(F.col("id").cast("string"), 6, "0")).alias("padded_id"),
    F.format_string("%s has %d chars", "product_code", F.length("product_code")).alias("described"),
).show(truncate=False)

## 5. Regular expressions: clean, extract, validate

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Strips non-digits from phones, keeps the last 10 digits, extracts the email domain, and validates the product code format.<br><br>
<b>Why it matters</b><br><code>regexp_replace</code> and <code>regexp_extract</code> handle what simple functions can't. In Python, write patterns as <b>raw strings</b> (<code>r"..."</code>) so backslashes reach Spark unchanged.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>digits</code> contains only numbers, <code>phone_10</code> is the last 10 digits, <code>email_domain</code> is lower-case (<code>example.com</code>, <code>corp.co.uk</code>, ...), and <code>valid_code</code> is <code>false</code> only for <code>bad code</code>. <code>sku_number</code> is the 5-digit group, or an empty string when there is no match.
</div>

In [0]:
cleaned = raw.select(
    "id",
    F.regexp_replace("phone", r"[^0-9]", "").alias("digits"),
    F.expr(r"right(regexp_replace(phone, '[^0-9]', ''), 10)").alias("phone_10"),
    F.lower(F.trim(F.regexp_extract("email", r"@(.+)$", 1))).alias("email_domain"),
    F.col("product_code").rlike(r"^SKU-[A-Z]{2}-\d{5}-[A-Z]+$").alias("valid_code"),
    F.regexp_extract("product_code", r"-(\d{5})-", 1).alias("sku_number"),
)
cleaned.show(truncate=False)

## 6. Edge case: no match, nulls and empty strings

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows what <code>regexp_extract</code>, <code>substring</code> and <code>split</code> return when the input doesn't match or is null.<br><br>
<b>Why it matters</b><br><code>regexp_extract</code> returns an <b>empty string</b>, not null, when there is no match, so <code>isNull()</code> checks miss it. Convert <code>''</code> to null explicitly with <code>nullif</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For <code>bad code</code>, the extract is <code>''</code> and <code>sku_or_null</code> is null. For the null email, every function returns null.
</div>

In [0]:
raw.select(
    "id", "product_code", "email",
    F.regexp_extract("product_code", r"-(\d{5})-", 1).alias("sku_raw"),
    F.expr(r"nullif(regexp_extract(product_code, '-(\\d{5})-', 1), '')").alias("sku_or_null"),
    F.split("email", "@")[0].alias("email_user"),
).show(truncate=False)

## 7. Searching, translate and fuzzy matching

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Finds substrings with <code>instr</code> and <code>contains</code>, maps characters with <code>translate</code>, and measures similarity with <code>levenshtein</code>.<br><br>
<b>Why it matters</b><br><code>translate</code> is a fast way to replace single characters (for example removing punctuation). <code>levenshtein</code> (edit distance) helps find near-duplicate names during data matching.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>at_pos</code> shows where <code>@</code> appears in each email, <code>no_dashes</code> removes dashes from codes, and <code>distance_to_amal</code> is 0 for <code>Amal</code> and small for similar spellings.
</div>

In [0]:
raw.select(
    "email",
    F.instr("email", "@").alias("at_pos"),
    F.col("email").contains(".ae").alias("is_uae"),
).show(truncate=False)

raw.select(F.translate("product_code", "-", "").alias("no_dashes")).show(truncate=False)

spark.createDataFrame([("Amal",), ("Amaal",), ("Aman",), ("John",)], "name STRING") \
    .select("name", F.levenshtein("name", F.lit("Amal")).alias("distance_to_amal")).show()

## 8. The same in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a typical cleaning query in Spark SQL.<br><br>
<b>Why it matters</b><br>Every function above has a SQL form with the same name. SQL is often more readable for long cleaning expressions.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Clean names, 10-digit phones and lower-case emails for all 5 rows.
</div>

In [0]:
raw.createOrReplaceTempView("raw_customers")
spark.sql(r"""
    SELECT id,
           initcap(trim(split(full_name, ',')[1]))                  AS first_name,
           initcap(trim(split(full_name, ',')[0]))                  AS last_name,
           right(regexp_replace(phone, '[^0-9]', ''), 10)           AS phone_10,
           lower(trim(email))                                       AS email
    FROM raw_customers
""").show(truncate=False)

## Under the hood

```
F.regexp_replace(F.trim("phone"), "[^0-9]", "")
        |
        v
 Catalyst expression tree:  RegExpReplace(StringTrim(phone), "[^0-9]", "")
        |
        v
 Whole-stage codegen: compiled into Java code working on UTF8String bytes, row by row
        |
        v
 No Python involved, no serialization, no shuffle
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> string functions work row by row inside each partition.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> built-in functions are expressions that Catalyst can optimize and code-generate.

A Python UDF doing the same work would send each row to a Python worker and back (lesson 20). With built-ins, everything stays in the JVM. Check the `Project` line in the plan below: you will see `regexp_replace` and `trim` as native expressions.

In [0]:
spark.range(1000).select(
    F.regexp_replace(F.trim(F.concat(F.lit(" +91-"), F.col("id").cast("string"))), "[^0-9]", "").alias("digits")
).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: whole column becomes null after concat</b><br>
<code>concat</code> returns null if any input is null. Use <code>concat_ws</code> or <code>coalesce</code> the inputs.<br><br>
<b>⛔ Problem: substring returns the wrong characters</b><br><code>substring</code> positions are <b>1-based</b>, while array indexes from <code>split(...)[i]</code> are <b>0-based</b>.<br><br>
<b>⛔ Problem: regex doesn't match or backslashes disappear</b><br>Use Python raw strings <code>r"\d+"</code>. Inside a SQL string in <code>F.expr</code>, backslashes may need doubling.<br><br>
**⛔ Problem: <code>isNull()</code> doesn't catch failed extracts**<br><code>regexp_extract</code> returns <code>''</code> on no match. Use <code>nullif(..., '')</code>.<br><br>
**⛔ Problem: <code>split</code> on <code>.</code> or <code>|</code> gives odd results**<br>The separator is a regex. Escape special characters: <code>split(col, r"\.")</code> or <code>r"\|"</code>.<br><br>
<b>⛔ Problem: case-sensitive comparisons miss matches</b><br>Normalize with <code>lower</code>/<code>upper</code> first, or use <code>ilike</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>concat</code> and <code>concat_ws</code>?</b><br>
<code>concat</code> joins values and returns null if any input is null. <code>concat_ws</code> takes a separator and skips null inputs, so it's usually safer for building names or keys.<br><br>

<b>🎤 2. How do you extract part of a string with a regex in Spark?</b><br>
<code>regexp_extract(col, pattern, group_index)</code> returns the captured group. It returns an empty string when there is no match, not null. <code>regexp_extract_all</code> returns all matches as an array.<br><br>

<b>🎤 3. How do you remove all non-numeric characters from a column?</b><br>
<code>regexp_replace(col, "[^0-9]", "")</code>.<br><br>

<b>🎤 4. Are <code>substring</code> positions 0-based or 1-based?</b><br>
1-based, like SQL. But array indexing after <code>split</code> is 0-based with <code>[i]</code>, while <code>element_at</code> is 1-based.<br><br>

<b>🎤 5. Why use built-in string functions instead of a Python UDF?</b><br>
Built-ins are Catalyst expressions that are optimized and code-generated in the JVM. A Python UDF serializes every row to a Python process and back, which is much slower and opaque to the optimizer.<br><br>

<b>🎤 6. How would you find near-duplicate names?</b><br>
Normalize first (trim, lower case, remove punctuation), then compare with <code>levenshtein</code> or <code>soundex</code>, usually within a blocking key such as postcode to avoid comparing every pair.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Column <code>code</code> contains values like <code>'ORD-2024-000123'</code>. Which expression returns <code>'000123'</code>?</b><br>
A. <code>substring(code, 10, 6)</code><br>
B. <code>regexp_extract(code, '-(\d+)$', 1)</code><br>
C. <code>split(code, '-')[2]</code><br>
D. All of the above<br>
<details><summary><b>Show answer</b></summary><b>D.</b> Positions in <code>substring</code> are 1-based (the digits start at character 10), the regex captures the digits after the last dash, and <code>split</code> produces <code>['ORD', '2024', '000123']</code> with 0-based indexing.</details><br><br>

<b>Q2. Which function returns NULL if any of its inputs is NULL?</b><br>
A. <code>concat_ws</code><br>
B. <code>concat</code><br>
C. <code>coalesce</code><br>
D. <code>nvl</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>concat</code> propagates nulls. <code>concat_ws</code> skips them.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>raw</code>:
<ol><li>Create <code>email_clean</code>: trimmed, lower case, and null if empty</li><li>Create <code>phone_10</code> (digits only, last 10 digits) and <code>country_code</code> (the digits before the last 10, or <code>91</code> when there are none)</li><li>Extract <code>country</code> and <code>colour</code> from <code>product_code</code> with a single regex (<code>SKU-&lt;country&gt;-&lt;5 digits&gt;-&lt;colour&gt;</code>), and null when it doesn't match</li><li>Build <code>display_name</code> as <code>Last, First</code> in title case</li><li>Count how many product codes are invalid</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
code_pattern = r"^SKU-([A-Z]{2})-\d{5}-([A-Z]+)$"

solution = (
    raw
    # helper column with digits only, dropped at the end
    .withColumn("digits", F.regexp_replace("phone", r"[^0-9]", ""))
    .select(
        "id",
        # 1. trimmed, lower-case email with '' turned into null
        F.expr("nullif(lower(trim(email)), '')").alias("email_clean"),
        # 2. last 10 digits, and whatever comes before them as the country code
        F.expr("right(digits, 10)").alias("phone_10"),
        F.expr("coalesce(nullif(left(digits, length(digits) - 10), ''), '91')").alias("country_code"),
        # 3. two capture groups from one pattern, '' -> null
        F.expr(f"nullif(regexp_extract(product_code, '{code_pattern}', 1), '')").alias("country"),
        F.expr(f"nullif(regexp_extract(product_code, '{code_pattern}', 2), '')").alias("colour"),
        # 4. 'Last, First' in title case
        F.concat_ws(
            ", ",
            F.initcap(F.trim(F.split("full_name", ",")[0])),
            F.initcap(F.trim(F.split("full_name", ",")[1])),
        ).alias("display_name"),
    )
)
solution.show(truncate=False)

# 5. invalid product codes
print("Invalid codes:", raw.filter(~F.col("product_code").rlike(code_pattern)).count())

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Normalize first: <code>trim</code>, <code>lower</code>/<code>upper</code>, <code>initcap</code></li><li><code>concat</code> propagates nulls, <code>concat_ws</code> skips them</li><li><code>substring</code> is 1-based, <code>split(...)[i]</code> is 0-based, <code>element_at</code> is 1-based</li><li><code>split</code> and <code>regexp_*</code> patterns are regular expressions. Use raw strings in Python</li><li><code>regexp_extract</code> returns <code>''</code> on no match. Wrap in <code>nullif(..., '')</code></li><li><code>rlike</code> validates formats, <code>regexp_replace</code> cleans, <code>regexp_extract</code> captures</li><li>Built-in string functions are native and fast. Avoid UDFs for text cleaning</li></ul>
</div>

| Task | Code |
|---|---|
| Clean | `F.lower(F.trim("c"))` |
| Join text | `F.concat_ws(" ", "a", "b")` |
| Slice | `F.substring("c", 1, 3)` |
| Split | `F.split("c", ",")[0]` |
| Remove chars | `F.regexp_replace("c", r"[^0-9]", "")` |
| Capture | `F.regexp_extract("c", r"(\d+)", 1)` |
| Validate | `F.col("c").rlike(r"^\d{5}$")` |
| Pad | `F.lpad("c", 6, "0")` |

## Git commit

```
git add 01_spark_basics/14_string_functions.ipynb
git commit -m "add 14 string functions notebook"
```